# import library

In [ ]:
import numpy as np
import pandas as pd
import os

# text
import nltk
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import words
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from pandarallel import pandarallel

#visualisation
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D # module 3D de Matplotlib
import seaborn as sns
import wordcloud
from PIL import Image
from wordcloud import WordCloud


#Modelisation
import sklearn # pip install scikit-learn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn import cluster, metrics
from sklearn import manifold, decomposition

# Option

In [ ]:
pd.set_option('display.max_columns', None) # affiche toutes les colonnes
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes dans toute leur largeur
#pd.set_option('display.max_rows', None)# Affiche les lignes dans toute leur largeu

# import data

In [ ]:
df_text = pd.read_csv("data/Flipkart/flipkart_com-ecommerce_sample_1050.csv")

# Explore data (text)
## Describe data

In [ ]:
df_text.columns

In [ ]:
df_text.info()

In [ ]:
df_text.head()

In [ ]:
df_text.dtypes.value_counts()


In [ ]:
df_text.describe()

In [ ]:
df_text["retail_price"].max()

In [ ]:
df_text.loc[df_text["retail_price"]>10000,:]["product_name"]

## Check Nan

In [ ]:
df_text.isna().sum()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(20,6))
sns.heatmap(df_text.isna())

## Check duplicated

In [ ]:
df_text.nunique()

In [ ]:
# Check duplicated
df_text.duplicated(subset = None, keep = 'first').unique() # pas de duplicated

In [ ]:
df_text.duplicated().sum()

In [ ]:
df_text.duplicated(subset='description').sum()

In [ ]:
for col in df_text.columns :
    n = df_text.duplicated(subset=col).sum()
    print(f"col:{col} -> duplicated :{n}")

# TEXT 
## Description

###  ANALYSE product_category

In [ ]:
df_text["product_category_tree"]

In [ ]:
# need to extract the first 20 line
df_text["product_category_tree"].str[0:25]


In [ ]:
# check the last 50 line
df_text["product_category_tree"].str[-50:-1]


In [ ]:
import re

In [ ]:
set(df_text["product_category_tree"].str[0:21])

In [ ]:
# categer libr ast, litteral eval fonction permet de ast.litteraleval, transfor string en liste si liste....

In [ ]:
# hypo une seul category par product
df_text["product_category_tree"][0][2:].split('>>')# pri le 1er element de la cellule, vire les 2 prems caract, split separateur donne list
df_text["product_category_tree"][0][2:].split('>>')[0].strip() # on veut que la cate principale et strip pour virer l espace

In [ ]:
df_text["category"] = df_text["product_category_tree"].apply(lambda x: x[2:].split('>>')[0].strip())
set(df_text["category"])

In [ ]:
list_category = set(df_text["category"])
list_category

In [ ]:
len(df_text)

In [ ]:
df_text.describe(include=object) # un tweet referencer 10 fois..need nettoyage

In [ ]:
len(df_text["product_name"].unique())

#### Create a df with only text col

In [ ]:
cols = ["product_name","description","category"]
df = df_text[cols]
df

#### ANALYSE COL DESCRIPTION

In [ ]:
df.description.apply(lambda i : len(i))

In [ ]:
tmp = df.description.str.len() # methode len issu de str
sns.boxplot(tmp) 

In [ ]:
sns.displot(tmp)

In [ ]:
df["_len_txt"] = tmp #underscore avant pour dire que cest nous qui avons creer
df

In [ ]:
df.sort_values('_len_txt').head(5) # 10 valeur les plus courtes

#### ANALYSE COL product name

In [ ]:
df.product_name.apply(lambda i : len(i))

In [ ]:
tmp2 = df.product_name.str.len() # methode len issu de str
sns.boxplot(tmp2) 

In [ ]:
sns.displot(tmp2)

In [ ]:
df["_len_product_name"] = tmp2 #underscore avant pour dire que cest nous qui avons creer
df

### Cleaning
#### Import library

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import words
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from pandarallel import pandarallel

In [ ]:
pandarallel.initialize(progress_bar = True, nb_workers=6) #verbose=1
sns.set()

In [ ]:
# chaque ligne de la col text correspond a un document
doc = df.description.sample(1)
doc = doc.values[0].lower()
doc

#### Tokenisation

In [ ]:
# example doc 1
tokenizer = nltk.RegexpTokenizer(r'\w+')
tokena = wordpunct_tokenize(doc)
tokena

In [ ]:
len(tokena)

In [ ]:
def display_tokens_info(tokens):
    """display info about corpus"""
    print(f"nb tokens {len(tokens)}, nb tokens uniques {len(set(tokens))}")
    print(tokens[:30])

In [ ]:
display_tokens_info(tokena)

In [ ]:
# another tokensizer
tokenizer2 = RegexpTokenizer(r"\w+")
tokens2 = tokenizer2.tokenize(doc)
display_tokens_info(tokens2) # certains mots ont disparu 43 et plus 49, mieux

In [ ]:
# apply to our short col
df2 = df[["description"]].copy()
df2.head(), df2.shape, df2.columns

#### Stop words

In [ ]:
stop_words = set(stopwords.words('english')) # dl stop word english
print(stop_words)

In [ ]:
def process_text1(doc, rejoin=False):
    """basic function of test processing"""
    #lower
    doc = doc.lower().strip()

    #tokemise4
    tokenizer = RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(doc)

    # stop word remove
    cleaned_token_list = [w for w in tokens if w not in stop_words]

    if rejoin :
        return " ".join(cleaned_token_list)
    
    return cleaned_token_list


In [ ]:
process_text1(doc)

#### Clean : remove numeric carcter, use is.alpha fonction, remove Stop words

In [ ]:
def process_text1(doc, rejoin=False,force_is_alpha=True):
    """basic function of test processing"""
    #lower
    doc = doc.lower().strip()

    #tokemise4
    tokenizer = RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(doc)

    # stop word remove
    cleaned_token_list = [w for w in tokens if w not in stop_words]
    
    if force_is_alpha :
        alpha_token =  [w for w in cleaned_token_list if w.isalpha()]
    else:
        alpha_token =  cleaned_token_list
    if rejoin :
        return " ".join(cleaned_token_list)
    
    return alpha_token



In [ ]:
process_text1(doc)

In [ ]:
tokens3 = process_text1(doc)
display_tokens_info(tokens3)

In [ ]:
# apply to every doc/row
df['tokenized'] = df.description.apply(lambda doc: process_text1(doc))
df.shape, df.head()

#### Stem and Lem
2 methodes pour uniformiser les mots
lemmentiser
stemming remove sufixe

In [ ]:
pd.Series(words.words()).sample(30) # dans nltk, list de mot anglais words.words
len(set(words.words())) # nb de mot dans dictionnaire anglais

In [ ]:
# uniformiser avec lower
eng_words = [i.lower() for i in words.words()]
print(eng_words[:20])

In [ ]:
trans = PorterStemmer()
eng_words_stem = [trans.stem(i) for i in eng_words]
display_tokens_info(eng_words_stem)
len(set(eng_words)) # output un peu moins

In [ ]:
def process_text2(doc, 
            rejoin=False, 
            lem_or_stem='stem',
            eng_words=eng_words_stem,
            force_is_alpha=True):


    #lower
    doc = doc.lower().strip()

    #tokemise4
    tokenizer = RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(doc)

    # stop word remove 
    cleaned_token_list = [w for w in tokens if w not in stop_words]

    # remove if not alphabetic
    if force_is_alpha :
        alpha_token =  [w for w in cleaned_token_list if w.isalpha()]
    else:
        alpha_token =  cleaned_token_list
    
    if lem_or_stem == 'lem':
        trans = WordNetLemmatizer()
        trans_test = [trans.lemmatize(i) for i in alpha_token]
    else:
        trans = PorterStemmer()
        trans_test = [trans.stem(i) for i in alpha_token]  

    # english words ne garde que mot englais
    if eng_words:
        eng_text = [i for i in trans_test if i in eng_words]
    else:
        eng_text = trans_test

    if rejoin :
        return " ".join(trans_test) # renvoie une chaine de caractere
    
    return eng_text

In [ ]:
process_text2(doc)

In [ ]:
display_tokens_info(process_text2(doc))

In [ ]:
df['tokenized'] = df.description.apply(lambda doc: process_text2(doc))

In [ ]:
df['tokenized_lem'] = df.description.apply(lambda doc: process_text2(doc,lem_or_stem='lem'))
df.shape, df.head()
# stem remove suffixe et prefixe

In [ ]:
df.tokenized.apply(lambda i : display_tokens_info(i))

In [ ]:
# freq token unique
df["nb_unique"] = df.tokenized.apply(lambda i : len(set(i)))
df["nb_unique"] 

### Creation corpus

In [ ]:
# ne marque que pour full join corpus
df['tokenized2'] = df.description.apply(lambda doc: process_text2(doc, rejoin=True))
raw_corpus = "".join(df.tokenized2.values)

In [ ]:
raw_corpus[0:100]

### representation graphique du corpus

In [ ]:
from collections import defaultdict

In [ ]:
len(raw_corpus.split())

In [ ]:
list_unique_corpus = np.unique(raw_corpus.split())
print (list_unique_corpus[5])

In [ ]:
len(list_unique_corpus)

In [ ]:
sum(np.array(raw_corpus.split())==list_unique_corpus[5])

In [ ]:
freq=[]
for i in list_unique_corpus:
    freq.append(sum(np.array(raw_corpus.split())==i))
freq

In [ ]:
#distribution de freq de mot
plt.hist(freq, bins=1000) # faire un bins par freq?
plt.xlim(0,20)
plt.show()

### Determiner liste de mot frequent, et oter les mots non informatifs

In [ ]:
df_tmp = pd.DataFrame({"mot_unique":list_unique_corpus,"occurence":freq})
df_tmp = df_tmp.sort_values(by = 'occurence', ascending=False)
df50 = df_tmp[0:50]

df50

In [ ]:
fig = plt.figure(figsize=(15,6))
plt.bar(df50["mot_unique"],height=df50["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,800)
plt.show()
# rs et I a virer

In [ ]:
# recherche mots >150 occu
df68 = df_tmp[df_tmp["occurence"]>150]
df68

In [ ]:
fig = plt.figure(figsize=(15,6))
plt.bar(df68["mot_unique"],height=df68["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,800)
plt.show()
# rs et I a virer

In [ ]:
set(df["category"])

In [ ]:
# Crer liste mot commun pas discriminant
list_mot_commun = list(df_tmp[df_tmp["occurence"]>150]["mot_unique"])
len(list_mot_commun)


In [ ]:
# uptade list
list_mot_commun.remove("mug"),list_mot_commun.remove("cm"),list_mot_commun.remove("color"),list_mot_commun.remove("watch"),list_mot_commun.remove("babi"),list_mot_commun.remove("materi"),
list_mot_commun.remove("cotton"),list_mot_commun.remove("girl"),list_mot_commun.remove("laptop"),list_mot_commun.remove("inch"),list_mot_commun.remove("ceram"),
list_mot_commun.remove("boy"),list_mot_commun.remove("india"),list_mot_commun.remove("dimens"),list_mot_commun.remove("men"),list_mot_commun.remove("skin"),list_mot_commun.remove("usb"),
list_mot_commun.remove("home"), list_mot_commun.remove("multicolor"),list_mot_commun.remove("light"),list_mot_commun.remove("coffe")


In [ ]:
list_mot_commun.append("key")

In [ ]:
# recherche mots frequent pas informatif
df100_150 = df_tmp[df_tmp["occurence"]<150][df_tmp["occurence"]>100]#["mot_unique"]
fig = plt.figure(figsize=(15,6))
plt.bar(df100_150["mot_unique"],height=df100_150["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,df100_150["occurence"].max())
plt.show()

In [ ]:
df100_150["mot_unique"]

In [ ]:
list_mot_commun.append("one"),list_mot_commun.append("discount"),list_mot_commun.append("deliveribuy"),list_mot_commun.append("also"),list_mot_commun.append("guarantebuy"),list_mot_commun.append("name"),
list_mot_commun.append("easi"),list_mot_commun.append("come"),
list_mot_commun.append("fit"),list_mot_commun.append("ye")

In [ ]:
list_mot_commun.append("onli")

In [ ]:
len(list_mot_commun)

In [ ]:
# recherche mots frequent pas informatif
df100_50 = df_tmp[df_tmp["occurence"]<100][df_tmp["occurence"]>50]
fig = plt.figure(figsize=(20,6))
plt.bar(df100_50["mot_unique"],height=df100_50["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,df100_50["occurence"].max())
plt.show()

In [ ]:
list_mot_commun.append("addit"),list_mot_commun.append("casual"),list_mot_commun.append("safe"),list_mot_commun.append("g"),
list_mot_commun.append("get"),list_mot_commun.append("like"),list_mot_commun.append("deliveriflipkart"),list_mot_commun.append("durabl"),list_mot_commun.append("year"),list_mot_commun.append("may"),
list_mot_commun.append("includ"),list_mot_commun.append("exclus"),list_mot_commun.append("add"),list_mot_commun.append("provid"),
list_mot_commun.append("id"),list_mot_commun.append("well")

In [ ]:
len(list_mot_commun)

In [ ]:
# recherche mots frequent pas informatif
df40_50 = df_tmp[df_tmp["occurence"]<50][df_tmp["occurence"]>40]
fig = plt.figure(figsize=(20,6))
plt.bar(df40_50["mot_unique"],height=df40_50["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,df40_50["occurence"].max())
plt.show()

In [ ]:
list_mot_commun.append("w"),list_mot_commun.append("sure"),list_mot_commun.append("item"),list_mot_commun.append("happi")


In [ ]:
# recherche mots frequent pas informatif
df40_30 = df_tmp[df_tmp["occurence"]<40][df_tmp["occurence"]>30]
fig = plt.figure(figsize=(20,6))
plt.bar(df40_30["mot_unique"],height=df40_30["occurence"])
plt.xticks(rotation = 90) 
plt.ylim(0,df40_30["occurence"].max())
plt.show()

In [ ]:
df40_30["mot_unique"][60:70]

In [ ]:
list_mot_commun.append("keep"),list_mot_commun.append("good"),list_mot_commun.append("check"),list_mot_commun.append("note"),
list_mot_commun.append("month"),list_mot_commun.append("new"),list_mot_commun.append("need"),list_mot_commun.append("summari"),list_mot_commun.append("e")

In [ ]:
# recherche mots frequent pas informatif
df20_30 = df_tmp[df_tmp["occurence"]<30][df_tmp["occurence"]>20]
fig = plt.figure(figsize=(20,6))
plt.bar(df20_30["mot_unique"][0:50],height=df20_30["occurence"][0:50])
plt.xticks(rotation = 90) 
plt.ylim(0,df20_30["occurence"].max())
plt.show()

In [ ]:
fig = plt.figure(figsize=(20,6))
plt.bar(df20_30["mot_unique"][50:100],height=df20_30["occurence"][50:100])
plt.xticks(rotation = 90) 
plt.ylim(0,df20_30["occurence"].max())
plt.show()

In [ ]:
df20_30["mot_unique"][-10:-1]

In [ ]:
list_mot_commun.append("find"),list_mot_commun.append("differ"),list_mot_commun.append("due"),list_mot_commun.append("similar"),
list_mot_commun.append("distinct"),list_mot_commun.append("say"),list_mot_commun.append("other"),list_mot_commun.append("know"),list_mot_commun.append("without"),
list_mot_commun.append("key"),list_mot_commun.append("two"),list_mot_commun.append("mean"),list_mot_commun.append("messag"),list_mot_commun.append("per"),list_mot_commun.append("offer"),
list_mot_commun.append("would"),list_mot_commun.append("deliverimaxima"),list_mot_commun.append("certif"),list_mot_commun.append("n"),list_mot_commun.append("full"),list_mot_commun.append("non"),

In [ ]:
# recherche mots frequent pas informatif
df20_10 = df_tmp[df_tmp["occurence"]<20][df_tmp["occurence"]>10]
fig = plt.figure(figsize=(20,6))
plt.bar(df20_10["mot_unique"][0:50],height=df20_10["occurence"][0:50])
plt.xticks(rotation = 90) 
plt.ylim(0,df20_10["occurence"].max())
plt.show()

In [ ]:
df20_10["mot_unique"][260:270] # les mots entre10 et 20 sont ultra informatif, je pense que ce sera difficil de les extraire dans le full text

In [ ]:
list_mot_commun.append("go"),list_mot_commun.append("gb"),list_mot_commun.append("ce"),list_mot_commun.append("necessari"),
list_mot_commun.append("ac"),list_mot_commun.append("simpl"),list_mot_commun.append("gentl"),list_mot_commun.append("b"),
list_mot_commun.append("purpos"),list_mot_commun.append("have"),list_mot_commun.append("compar"),list_mot_commun.append("even"),
list_mot_commun.append("r"),list_mot_commun.append("export"),list_mot_commun.append("take"),
list_mot_commun.append("na"),list_mot_commun.append("standard"),list_mot_commun.append("must"),list_mot_commun.append("send"),
list_mot_commun.append("stand"),list_mot_commun.append("amaz"),list_mot_commun.append("excel"),list_mot_commun.append("super"),
list_mot_commun.append("perfectli"),list_mot_commun.append("caus"),list_mot_commun.append("kind"),list_mot_commun.append("c"),list_mot_commun.append("approx")

In [ ]:
len(list_mot_commun),list_mot_commun

In [ ]:
list_mot_commun.append("thi")

#### Liste mot unique, ou inf a 5

In [ ]:
len(df_tmp[df_tmp["occurence"]<5])

In [ ]:
list_token_5 = list(df_tmp[df_tmp["occurence"]<6][df_tmp["occurence"]>1]["mot_unique"])
len(list_token_5),list_token_5

In [ ]:
list_token_used_unique = list(df_tmp[df_tmp["occurence"]<2]["mot_unique"])
list_token_used_unique

#### par categorie, 1/ compte freq des mot, regarde quel mot sont prst dans toutes les categories.. donc a virer

In [ ]:
df.columns

In [ ]:
# create un df par categorie
df_temp2 = df[['category','product_name','tokenized2']]
df_cat = df_temp2.groupby(['category'], as_index=False).agg({'tokenized2': ' '.join,'product_name': ' '.join})

In [ ]:
list_category

In [ ]:
str(df['tokenized2'][df['category']== 'Watches'])

In [ ]:
def visu_freq_unique(dataframe,category):
    """ Fonction qui retourne un hist avec la freq unique des token par category"""
    # create un df par categorie
    df_temp2 = dataframe[['category','product_name','tokenized2']]
    df_cat = df_temp2.groupby(['category'], as_index=False).agg({'tokenized2': ' '.join,'product_name': ' '.join})

    # creation corpus
    corpus_cat = str(df_cat['tokenized2'][df_cat['category']== category])

    # Creation of unique token 
    list_unique_cat = np.unique(corpus_cat.split())

    # Calcule la freq de chaque token unique par categorie
    freq_cat = []
    for i in list_unique_cat:
        freq_cat.append(sum(np.array(corpus_cat.split())==i))

    # Visu each category
    # Create df
    df_tmp_local = pd.DataFrame({"mot_unique":list_unique_cat,"occurence":freq_cat})
    df_tmp_local = df_tmp_local.sort_values(by = 'occurence', ascending=False)

    # histogram
    df60 = df_tmp_local[0:60]

    fig = plt.figure(figsize=(15,6))
    plt.bar(df60["mot_unique"],height=df60["occurence"])
    plt.xticks(rotation = 90) 
    plt.ylim(0,df60["occurence"].max())
    plt.title("Distribution frequence token unique pour la categorie : " + str(category))

    return plt.show()

In [ ]:
visu_freq_unique(df,'Watches')

In [ ]:
for el in list_category: 
    visu_freq_unique(df_cat,el)

In [ ]:
# need compare freq de chaque cate

In [ ]:
def more_freq_token_unique(df,row):
    """ Fonction qui retourn une liste des tokens uniques par category"""
    # creation corpus
    corpus_cat = df['tokenized2'][row]
    
    # Creation of unique token 
    list_unique_cat = np.unique(corpus_cat.split())

    # Calcule la freq de chaque token unique par categorie
    freq_cat = []
    for i in list_unique_cat:
        freq_cat.append(sum(np.array(corpus_cat.split())==i))

    # Create df
    df_tmp = pd.DataFrame({"mot_unique":list_unique_cat,"occurence":freq_cat})
    df_tmp = df_tmp.sort_values(by = 'occurence', ascending=False)

    # prend la liste de mot les plus freq
    
    return list(df_tmp['mot_unique'])

In [ ]:
others = df['tokenized2'][1:7]
others

In [ ]:
more_freq_token_unique(df_cat,0)

In [ ]:
# check the diff, keep difference cat 0 vs all others
list_diff_cat0_vs_all = list(set(more_freq_token_unique(df_cat,0))-set(more_freq_token_unique(df_cat,1)+more_freq_token_unique(df_cat,2)
+more_freq_token_unique(df_cat,3)+more_freq_token_unique(df_cat,4)+more_freq_token_unique(df_cat,5)+more_freq_token_unique(df_cat,6)))

In [ ]:
# Creer liste de mot present dans toute les categories
list_mot_non_discrimant = list(set(more_freq_token_unique(df_cat,0))
    &set(more_freq_token_unique(df_cat,1))
    &set(more_freq_token_unique(df_cat,2))
    &set(more_freq_token_unique(df_cat,3))
    &set(more_freq_token_unique(df_cat,4))
    &set(more_freq_token_unique(df_cat,5))
    &set(more_freq_token_unique(df_cat,6)))

In [ ]:
len(list_mot_non_discrimant)

In [ ]:
def process_text3(doc, 
            rejoin=False, 
            lem_or_stem='stem',
            list_rare_word=None,
            min_len_word=3,
            force_is_alpha=True):
    
    # list unique words
    if not list_rare_word:
        list_rare_word = []
    
    #lower
    doc = doc.lower().strip()

    #tokemise4
    tokenizer = RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(doc)

    # stop word
    cleaned_tokwn_list = [w for w in tokens if w not in stop_words] # clean les token en enlevant les stopword
    non_rare_token = [w for w in cleaned_tokwn_list if w not in list_rare_word]# ote les mots rares
    more_than_N = [w for w in non_rare_token if len(w) >= min_len_word] # ne garde que les mots superieur a min len world
    
    if force_is_alpha :
        alpha_token =  [w for w in more_than_N if w.isalpha()] # selectionne les token 100 alphabetique, sans nombre
    else:
        alpha_token =  more_than_N

    if lem_or_stem == 'lem':
        trans = WordNetLemmatizer()
        trans_test = [trans.lemmatize(i) for i in alpha_token]
    else:
        trans = PorterStemmer()
        trans_test = [trans.stem(i) for i in alpha_token]

    if rejoin :
        return " ".join(alpha_token) # renvoie une chaine de caractere
    
    return trans_test

In [ ]:
# update fonction en enlevant mot les plus freq, non discriminant et les tokens unique
def process_text3(doc, 
            rejoin=False, 
            lem_or_stem='stem',
            eng_words=eng_words_stem,
            list_no_discrim_word=list_mot_non_discrimant,
            list_more_freq = list_mot_commun,
            list_rare_word=list_token_used_unique,
            min_len_word=3,
            force_is_alpha=True):
    """ 
    Fonction prend en argument un document
    et retourne une liste de token nettoye
    sans les mot frequent, non discriminant, non rare, seulement caractere alphabetique
    """
    #lower
    doc = doc.lower().strip()

    #tokemise4
    tokenizer = RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(doc)
    
    # remove if not alphabetic
    if force_is_alpha :
        alpha_token =  [w for w in tokens if w.isalpha()]
    else:
        alpha_token =  tokens
    
    if lem_or_stem == 'lem':
        trans = WordNetLemmatizer()
        trans_test = [trans.lemmatize(i) for i in alpha_token]
    else:
        trans = PorterStemmer()
        trans_test = [trans.stem(i) for i in alpha_token]  

    # english words ne garde que mot englais
    if eng_words:
        eng_text = [i for i in trans_test if i in eng_words]
    else:
        eng_text = trans_test

    # stop word remove , remove commun word
    cleaned_token_list = [w for w in eng_text if w not in stop_words]
    discrim_token = [w for w in cleaned_token_list if w not in list_no_discrim_word]
    not_common_token = [w for w in discrim_token if w not in list_more_freq]# ote les mots commun
    
    # remove unique or rare
    non_rare_token = [w for w in not_common_token if w not in list_rare_word]# ote les mots rares
    more_than_N = [w for w in non_rare_token if len(w) >= min_len_word] # ne garde que les mots superieur a min len world

    if rejoin :
        return " ".join(more_than_N) # renvoie une chaine de caractere
    
    return more_than_N

In [ ]:
# test fonction sur un seul doc
token4 = process_text3(doc) # fonction prend en arg un doc, et re

In [ ]:
display_tokens_info(token4)

In [ ]:
len(process_text3(df["description"][0])),len(np.unique(process_text3(df["description"][0]))), np.unique(process_text3(df["description"][0]) )

In [ ]:
# update cleaning col
df['tokenized'] = df.description.apply(lambda doc: process_text3(doc))

In [ ]:
# ne marque que pour full join corpus
df['tokenized2'] = df.description.apply(lambda doc: process_text3(doc, rejoin=True))
raw_corpus = "".join(df.tokenized2.values)

In [ ]:
def wordcloud(dataframe,category):
    """ Fonction qui retourne un hist avec la freq unique des token par category"""
    # create un df par categorie
    df_temp2 = dataframe[['category','product_name','tokenized2']]
    df_cat = df_temp2.groupby(['category'], as_index=False).agg({'tokenized2': ' '.join,'product_name': ' '.join})

    # creation corpus
    corpus_cat = str(df_cat['tokenized2'][df_cat['category']== category])

    # Creation of unique token 
    list_unique_cat = np.unique(corpus_cat.split())

    # Calcule la freq de chaque token unique par categorie
    freq_cat = []
    for i in list_unique_cat:
        freq_cat.append(sum(np.array(corpus_cat.split())==i))

    # visu card
    wordcloud = WordCloud(background_color='white',stopwords=[],max_words=40).generate("".join(corpus_cat))
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title("Distribution token unique pour la categorie : " + str(category))

    return plt.show()

In [ ]:
wordcloud(df,df["category"][0])

In [ ]:
# Check if common and rare word are removed
for el in list_category:
    wordcloud(df,el)

In [ ]:
# Check if common and rare word are removed
for el in list_category:
    visu_freq_unique(df,el)

In [ ]:
# need to remove color because to freq and everywhere
list_mot_commun.append("color")

In [ ]:
# update
df['tokenized'] = df.description.apply(lambda doc: process_text3(doc))
df['tokenized2'] = df.description.apply(lambda doc: process_text3(doc, rejoin=True))
raw_corpus = "".join(df.tokenized2.values)

In [ ]:
# Check if common and rare word are removed
for el in list_category:
    wordcloud(df,el)

In [ ]:
# Check if common and rare word are removed
for el in list_category:
    visu_freq_unique(df,el)

In [ ]:
# crer un corpus de 1050 doc, un doc = un string, list de string
corpus = df.tokenized2.values.tolist()
corpus
# ok bon format pour vectorizer

## Classic - bag of word : Transfo des data string 
### CountVectorizer


In [ ]:
# Equivalent to CountVectorizer followed by TfidfTransformer.
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF
from sklearn.pipeline import Pipeline

import scipy.sparse

In [ ]:
# vectorizer Convert a collection of raw documents to a matrix of count vectorizer features.
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
vectorizer.get_feature_names_out()
print(X.shape)

In [ ]:
print(X.toarray())

In [ ]:
X #shape ok row = doc, col = unique token

In [ ]:
df_count_vect = pd.DataFrame.sparse.from_spmatrix(X, columns=vectorizer.get_feature_names_out())
df_count_vect

In [ ]:
vectorizer2 = CountVectorizer(analyzer='word', ngram_range=(2, 2))
X2 = vectorizer2.fit_transform(corpus)
vectorizer2.get_feature_names_out()

In [ ]:
print(X2.toarray())

In [ ]:
df_count_vect2 = pd.DataFrame.sparse.from_spmatrix(X2, columns=vectorizer2.get_feature_names_out())
df_count_vect2

### TI DF
poids=frequence du n-gram×idf(n-gramme)
The goal of using tf-idf instead of the raw frequencies of occurrence of a token in a given document is to scale down the impact of tokens that occur very frequently in a given corpus and that are hence empirically less informative than features that occur in a small fraction of the training corpus.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_IDF = tfidf_vectorizer.fit_transform(corpus)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
print(X_IDF.shape) # ok meem shape qu avec vectorize


In [ ]:
# creer un df tfidf
df_count_tfidf = pd.DataFrame.sparse.from_spmatrix(X_IDF, columns=tfidf_feature_names)
df_count_tfidf

### Reduction des dimensions
#### ACP (Mieux que LAD)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn import decomposition

In [ ]:
# need format to array
X_cvect = X.toarray()
X_tidf = X_IDF.toarray()

In [ ]:
# Create model
pca = decomposition.PCA(n_components=2)
# Fit countvectorize 
pca.fit(X_cvect)
#print (pca.explained_variance_ratio_.cumsum()) # dans l autre sesn
X_trans = pca.transform(X_cvect)
print(pca.explained_variance_ratio_)

In [ ]:
X_trans

In [ ]:
list(list_category)[0]

In [ ]:
df["cat_number"] = 0
for i in range(7):
        df.loc[df_text["category"]== list(list_category)[i], "cat_number"] = i
set(df["cat_number"])

In [ ]:
df_X_trans = pd.DataFrame(X_trans)
df_X_trans

In [ ]:
df_X_trans["category_class"] = df["cat_number"]

In [ ]:
# visu
ax = plt.figure()
scatter = plt.scatter(X_trans[:,0],X_trans[:,1], c = df_X_trans["category_class"], cmap='Set1' )
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.xlim(-2,4)
plt.ylim(-5,7)
plt.title("PCA CountVectorize",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels = list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.show()

In [ ]:
# Fit TFIDF 
pca.fit(X_tidf)
# print (pca.explained_variance_ratio_.cumsum())
print (pca.explained_variance_ratio_)
X_IDF_trans = pca.transform(X_tidf)
#Transfo into df
df_X_IDF_trans = pd.DataFrame(X_IDF_trans)
df_X_IDF_trans["category_class"] = df["cat_number"]

# visu
ax = plt.figure()
plt.title("PCA TF-IDF",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels=list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.scatter(X_IDF_trans[:,0],X_IDF_trans[:,1],c = df_X_IDF_trans["category_class"], cmap='Set1' )
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.show()

#### T-SNE

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
# test 1
X_embedded_vect = TSNE(n_components=2, learning_rate='auto',
                init='pca', perplexity=100, random_state = 2023).fit_transform(X_cvect)
X_embedded_vect.shape


In [ ]:
X_embedded_vect

In [ ]:
import time

# Calcul Tsne, détermination des clusters et calcul ARI entre vrais catégorie et n° de clusters
def ARI_fct(features) :
    time1 = time.time()
    num_labels=len(list_category)
    tsne = manifold.TSNE(n_components=2, perplexity=20, n_iter=2000, 
                                 init='random', learning_rate='auto', random_state=42)
    X_tsne = tsne.fit_transform(features)
    
    # Détermination des clusters à partir des données après Tsne 
    cls = cluster.KMeans(n_clusters=num_labels, n_init='auto', random_state=42)
    cls.fit(X_tsne)
    y_cat_num = df["cat_number"]
    ARI = np.round(metrics.adjusted_rand_score(y_cat_num, cls.labels_),4)
    time2 = np.round(time.time() - time1,0)
    print("ARI : ", ARI, "time : ", time2)
    
    return ARI, X_tsne, cls.labels_


def TSNE_visu_fct(X_tsne, y_cat_num, labels, ARI,type) :
    fig = plt.figure(figsize=(15,6))
    ax = fig.add_subplot(121)
    scatter = ax.scatter(X_tsne[:,0],X_tsne[:,1], c=y_cat_num, cmap='Set1')
    plt.xlabel ("t-SNE1")
    plt.ylabel ("t-SNE2")
    plt.title(str(type) + ' : t-SNE, par catégorie',fontweight="bold",fontsize=20)
    
    ax = fig.add_subplot(122)
    scatter = ax.scatter(X_tsne[:,0],X_tsne[:,1], c=labels, cmap='Set1')
    ax.legend(handles=scatter.legend_elements()[0], labels=list_category, loc="best", title='Clusters',bbox_to_anchor=(1.05, 1), borderaxespad=0.0) #set(labels)
    plt.title('par clusters',fontweight="bold",fontsize=20)
    plt.xlabel ("t-SNE1")
    plt.show()
    print("ARI : ", ARI)

In [ ]:
# We want to get TSNE embedding with 2 dimensions
n_components = 2
tsne = TSNE(n_components=2, learning_rate='auto',
                init='pca', perplexity=100, random_state = 2023)
tsne_result = tsne.fit_transform(X_cvect)
tsne_result.shape # (1050, 2)

# Plot the result of our TSNE with the label color coded
tsne_result_df = pd.DataFrame({'tsne_1': tsne_result[:,0], 'tsne_2': tsne_result[:,1], 'label': df["cat_number"]})
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=120,alpha=0.4, cmap='Set1')
lim = (X_embedded_vect.min()-5, X_embedded_vect.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title('t-TSNE méthode countVectorosizer')  
plt.show()
# couleur, joue sur hyperparamt de la tsne, genere pls fois a la mano pour voir ce que ca change et voir si mieux

In [ ]:
# We want to get TSNE embedding with 2 dimensions
tsne_result_tidf = tsne.fit_transform(X_tidf)
tsne_result_tidf.shape

# create df
tsne_result_df_tdif = pd.DataFrame({'tsne_1': tsne_result_tidf[:,0], 'tsne_2': tsne_result_tidf[:,1], 'label': df["cat_number"]})

# Plot the result of our TSNE with the label color coded
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df_tdif, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_tidf.min(), tsne_result_tidf.max())
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title('t-TSNE méthode TF-IDF')  
plt.show()

### Clustering kmeans, k=7

In [ ]:
# import librairy
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler,StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score

In [ ]:
# Initialize the clusterer with n_clusters value and a random generator
model1_kmn7 = KMeans(n_clusters= 7, n_init="auto", random_state=10)

# fit
y_labels_model1_kmn3_cvect = model1_kmn7.fit_predict(tsne_result_df)
y_labels_model1_kmn3_cvect = model1_kmn7.fit_predict(tsne_result_tidf)

In [ ]:
y_labels_model1_kmn3_cvect

In [ ]:
# 3 cluster
y_cvect_km7 = model1_kmn7.fit(tsne_result)
y_tfidf_km7 = model1_kmn7.fit(tsne_result_tidf)

In [ ]:
tsne_result_df['Labels_KMN7'] = pd.DataFrame(y_cvect_km7.labels_)
tsne_result_df_tdif['Labels_KMN7'] = pd.DataFrame(y_tfidf_km7.labels_)
tsne_result_df['Labels_KMN7'].unique(),tsne_result_df_tdif['Labels_KMN7'].unique()

In [ ]:
tsne_result_df

In [ ]:
tsne_result_df_tdif

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='Labels_KMN7', data=tsne_result_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result.min(), tsne_result.max())
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title('t-TSNE méthode countVectorosizer')  
plt.show()


In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result.min(), tsne_result.max())
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='Labels_KMN7', data=tsne_result_df_tdif, ax=ax,s=120,alpha=1)
lim = (tsne_result_tidf.min(), tsne_result_tidf.max())
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df_tdif, ax=ax,s=120,alpha=1)
lim = (tsne_result_tidf.min(), tsne_result_tidf.max())
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
ARI_cvect, X_tsne_cvect, labels_cvect = ARI_fct(X_cvect) # doit trouver entre 0.4 et 0.5 a revoir

In [ ]:
ARI_tidf, X_tsne_tidf, labels_tidf = ARI_fct(X_tidf) # doit trouver entre 0.4 et 0.5 a revoir

In [ ]:
len(labels_cvect)

In [ ]:
# Visu count vectorizer
TSNE_visu_fct(X_tsne_cvect, df["cat_number"], labels_cvect, ARI_cvect,"Count Vectorizer") 

In [ ]:
# Visu TFIDF
TSNE_visu_fct(X_tsne_tidf, df["cat_number"], labels_tidf, ARI_tidf, "TF-IDF")

## WORD2VEC

In [ ]:
import tensorflow as tf


In [ ]:
import keras
from tensorflow import keras

In [ ]:
import tensorflow.keras
#from tensorflow import keras
from tensorflow.keras import backend as K

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import metrics as kmetrics
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import gensim

### Création du modèle Word2Vec

In [ ]:
w2v_size=300
w2v_window=5
w2v_min_count=1
w2v_epochs=100
maxlen = 192 # adapt to length of sentences
sentences = corpus #df['tokenizer2'].to_list()
sentences2 = [gensim.utils.simple_preprocess(text) for text in sentences] # from list of doc transfo each list into a list of word (liste imbriquee)

In [ ]:
max([len(sentence) for sentence in sentences2])

In [ ]:
# Création et entraînement du modèle Word2Vec
print("Build & train Word2Vec model ...")
w2v_model = gensim.models.Word2Vec(min_count=w2v_min_count, window=w2v_window,
                                                vector_size=w2v_size,
                                                seed=42,
                                                workers=1)
#                                                workers=multiprocessing.cpu_count())
w2v_model.build_vocab(sentences2)
w2v_model.train(sentences2, total_examples=w2v_model.corpus_count, epochs=w2v_epochs)
model_vectors = w2v_model.wv
w2v_words = model_vectors.index_to_key
print("Vocabulary size: %i" % len(w2v_words))
print("Word2Vec trained")

In [ ]:
# Préparation des sentences (tokenization) transfo en array de phrase et compte le nomre d occurense des tokens
print("Fit Tokenizer ...")
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences2)
x_sentences = pad_sequences(tokenizer.texts_to_sequences(sentences2),
                                                     maxlen=maxlen,
                                                     padding='post') 
                                                   
num_words = len(tokenizer.word_index) + 1
print("Number of unique words: %i" % num_words)

In [ ]:
x_sentences

### Création de la matrice d'embedding

In [ ]:
# Création de la matrice d'embedding

print("Create Embedding matrix ...")
w2v_size = 300
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
embedding_matrix = np.zeros((vocab_size, w2v_size))
i=0
j=0
    
for word, idx in word_index.items():
    i +=1
    if word in w2v_words:
        j +=1
        embedding_vector = model_vectors[word]
        if embedding_vector is not None:
            embedding_matrix[idx] = model_vectors[word]
            
word_rate = np.round(j/i,4)
print("Word embedding rate : ", word_rate)
print("Embedding matrix: %s" % str(embedding_matrix.shape))

### Création du modèle d'embedding

In [ ]:
# Création du modèle

input = Input(shape=(len(x_sentences),maxlen),dtype='float64')
word_input = Input(shape=(maxlen,),dtype='float64')  
word_embedding = Embedding(input_dim=vocab_size,
                         output_dim=w2v_size,
                         weights = [embedding_matrix],
                         input_length=maxlen)(word_input)
word_vec = GlobalAveragePooling1D()(word_embedding)  
embed_model = Model([word_input],word_vec)

embed_model.summary()

### Exécution du modèle

In [ ]:
embeddings = embed_model.predict(x_sentences)
embeddings.shape

### Reduction des dimensions
#### ACP 

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn import decomposition

In [ ]:
# Create model alrdeady down pca = decomposition.PCA(n_components=2)
# Fit wor2sec
pca.fit(embeddings)
#print (pca.explained_variance_ratio_.cumsum()) # dans l autre sesn
X_trans_w2s = pca.transform(embeddings)
print(pca.explained_variance_ratio_)

In [ ]:
df_X_trans_w2s= pd.DataFrame(X_trans_w2s)
df_X_trans_w2s

In [ ]:
# visu
ax = plt.figure()
plt.title("PCA Word2sec",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels=list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.scatter(X_trans_w2s[:,0],X_trans_w2s[:,1], c = df["cat_number"] , cmap='Set1') 
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.show()

#### T-SNE

In [ ]:
# test 1
X_embedded_w2s = TSNE(n_components=2, learning_rate='auto',
                init='pca', perplexity=100, random_state = 2023).fit_transform(embeddings)
X_embedded_w2s.shape
tsne_result_w2s = tsne.fit_transform(embeddings)
tsne_result_w2s.shape # (1050, 2)

In [ ]:
# Plot the result of our TSNE with the label color coded
tsne_result_w2s_df = pd.DataFrame({'tsne_1': tsne_result_w2s[:,0], 'tsne_2': tsne_result_w2s[:,1], 'label': df["cat_number"]})
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_w2s_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_w2s.min()-5, tsne_result_w2s.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title("t-SNE WORD2SEC")
# couleur, joue sur hyperparamt de la tsne, genere pls fois a la mano pour voir ce que ca change et voir si mieux

### Clustering kmeans, k=7

In [ ]:
# Initialize the clusterer with n_clusters value and a random generator :model1_kmn7 = KMeans(n_clusters= 7, n_init="auto", random_state=10)
# fit
y_labels_model1_kmn3_w2s = model1_kmn7.fit_predict(tsne_result_w2s_df)

In [ ]:
y_labels_model1_kmn3_w2s

In [ ]:
# 3 cluster
y_w2s_km7 = model1_kmn7.fit(tsne_result_w2s)
y_w2s_km7

In [ ]:
tsne_result_w2s_df['Labels_KMN7'] = pd.DataFrame(y_w2s_km7.labels_)
tsne_result_w2s_df['Labels_KMN7'].unique()

In [ ]:
tsne_result_w2s_df

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='Labels_KMN7', data=tsne_result_w2s_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_w2s.min()-5, tsne_result_w2s.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_w2s_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_w2s.min(), tsne_result_w2s.max())
ax.set_xlim(-25,+25)
ax.set_ylim(-22,+22)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
ARI_w2s, tsne_result_w2s, labels_w2s = ARI_fct(embeddings)

In [ ]:
TSNE_visu_fct(tsne_result_w2s, df["cat_number"], labels_w2s, ARI_w2s, "Word 2 VEC")

## BERT

In [ ]:
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras import backend as K

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import metrics as kmetrics
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

# Bert
import os
import transformers
from transformers import *

os.environ["TF_KERAS"]='1'

import tensorflow_hub as hub

In [ ]:
bert_layer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", trainable=True)

In [ ]:
print(tf.__version__)
print(tensorflow.__version__)
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

### Fonctions communes

In [ ]:
# Fonction de préparation des sentences
def bert_inp_fct(sentences, bert_tokenizer, max_length) :
    input_ids = []
    token_type_ids = []
    attention_mask = []
    bert_inp_tot = []

    for sent in sentences:
        bert_inp = bert_tokenizer.encode_plus(sent,
                                              add_special_tokens = True,
                                              max_length = max_length,
                                              padding='max_length',
                                              return_attention_mask = True, 
                                              return_token_type_ids=True,
                                              truncation=True,
                                              return_tensors="tf")
    
        input_ids.append(bert_inp['input_ids'][0])
        token_type_ids.append(bert_inp['token_type_ids'][0])
        attention_mask.append(bert_inp['attention_mask'][0])
        bert_inp_tot.append((bert_inp['input_ids'][0], 
                             bert_inp['token_type_ids'][0], 
                             bert_inp['attention_mask'][0]))

    input_ids = np.asarray(input_ids)
    token_type_ids = np.asarray(token_type_ids)
    attention_mask = np.array(attention_mask)
    
    return input_ids, token_type_ids, attention_mask, bert_inp_tot
    

# Fonction de création des features
def feature_BERT_fct(model, model_type, sentences, max_length, b_size, mode='HF') :
    batch_size = b_size
    batch_size_pred = b_size
    bert_tokenizer = AutoTokenizer.from_pretrained(model_type)
    time1 = time.time()

    for step in range(len(sentences)//batch_size) :
        idx = step*batch_size
        input_ids, token_type_ids, attention_mask, bert_inp_tot = bert_inp_fct(sentences[idx:idx+batch_size], 
                                                                      bert_tokenizer, max_length)
        
        if mode=='HF' :    # Bert HuggingFace
            outputs = model.predict([input_ids, attention_mask, token_type_ids], batch_size=batch_size_pred)
            last_hidden_states = outputs.last_hidden_state

        if mode=='TFhub' : # Bert Tensorflow Hub
            text_preprocessed = {"input_word_ids" : input_ids, 
                                 "input_mask" : attention_mask, 
                                 "input_type_ids" : token_type_ids}
            outputs = model(text_preprocessed)
            last_hidden_states = outputs['sequence_output']
             
        if step ==0 :
            last_hidden_states_tot = last_hidden_states
            last_hidden_states_tot_0 = last_hidden_states
        else :
            last_hidden_states_tot = np.concatenate((last_hidden_states_tot,last_hidden_states))
    
    features_bert = np.array(last_hidden_states_tot).mean(axis=1)
    
    time2 = np.round(time.time() - time1,0)
    print("temps traitement : ", time2)
     
    return features_bert, last_hidden_states_tot

### 'bert-base-uncased'

In [ ]:
len(corpus)

In [ ]:
df["_len_txt"] = df.tokenized.apply(lambda i : len(i))
df["_len_txt"].max()

In [ ]:
max_length = 192
batch_size = 1050
model_type = 'bert-base-uncased'
model_bert = TFAutoModel.from_pretrained(model_type)
sentences = corpus.copy() # une sentence = un doc

In [ ]:
# Création des features
features_bert, last_hidden_states_tot = feature_BERT_fct(model_bert, model_type, sentences, max_length, batch_size, mode='HF')

#### Reduction des dimensions
##### ACP 

In [ ]:
# Create model alrdeady down pca = decomposition.PCA(n_components=2)
# Fit wor2sec
pca.fit(features_bert)
#print (pca.explained_variance_ratio_.cumsum()) # dans l autre sesn
X_trans_bert = pca.transform(features_bert)
print(pca.explained_variance_ratio_)

In [ ]:
df_X_trans_bert= pd.DataFrame(X_trans_bert)
df_X_trans_bert

In [ ]:
# visu
ax = plt.figure()
plt.title("PCA BERT-base",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels=list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.scatter(df_X_trans_bert[0],df_X_trans_bert[1], c = df["cat_number"], cmap='Set1' ) 
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.xlim(-5,5)
plt.show()

##### T-SNE

In [ ]:
# test 1
X_embedded_bert = TSNE(n_components=2, learning_rate='auto',
                init='pca', perplexity=100, random_state = 2023).fit_transform(features_bert)
X_embedded_bert.shape
# We want to get TSNE embedding with 2 dimensions
# n_components = 2
# tsne = TSNE(n_components=2, learning_rate='auto', init='pca', perplexity=100, random_state = 2023)
tsne_result_bert = tsne.fit_transform(features_bert)
tsne_result_bert.shape # (1050, 2)


In [ ]:
# Plot the result of our TSNE with the label color coded
tsne_result_bert_df = pd.DataFrame({'tsne_1': tsne_result_bert[:,0], 'tsne_2': tsne_result_bert[:,1], 'label': df["cat_number"]})
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_bert_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_bert.min()-5, tsne_result_bert.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title("t-SNE BERT")

In [ ]:
TSNE_visu_fct(tsne_result_w2s, df["cat_number"], labels_w2s, ARI_w2s, "Word 2 VEC")

#### Clustering kmeans, k=7

In [ ]:
# Initialize the clusterer with n_clusters value and a random generator :model1_kmn7 = KMeans(n_clusters= 7, n_init="auto", random_state=10)
# fit
y_labels_model1_kmn3_bert = model1_kmn7.fit_predict(tsne_result_bert_df)

In [ ]:
y_labels_model1_kmn3_bert

In [ ]:
# 3 cluster
y_bert_km7 = model1_kmn7.fit(tsne_result_bert)
y_bert_km7

In [ ]:
tsne_result_bert_df['Labels_KMN7'] = pd.DataFrame(y_bert_km7.labels_)
tsne_result_bert_df['Labels_KMN7'].unique()

In [ ]:
tsne_result_bert_df

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='Labels_KMN7', data=tsne_result_bert_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_bert.min()-5, tsne_result_bert.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_bert_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_bert.min()-5, tsne_result_bert.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
ARI_bert, X_tsne_bert, labels_bert = ARI_fct(features_bert)

In [ ]:
TSNE_visu_fct(X_tsne_bert, df["cat_number"], labels_bert, ARI_bert, "BERT-base")

### BERT hub Tensorflow

In [ ]:
import tensorflow_hub as hub

# Guide sur le Tensorflow hub : https://www.tensorflow.org/text/tutorials/classify_text_with_bert
model_url = 'https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4'

In [ ]:
bert_layer2 = hub.KerasLayer(model_url, trainable=True)

In [ ]:
model = bert_layer2
features_bert2, last_hidden_states_tot = feature_BERT_fct(model, model_type, sentences, max_length, batch_size, mode='TFhub')

In [ ]:
ARI_bert2, X_tsne_bert2, labels_bert2 = ARI_fct(features_bert2)

In [ ]:
TSNE_visu_fct(X_tsne_bert2, df["cat_number"], labels_bert2, ARI_bert2, "BERT-hub tensorflow")

## USE - Universal Sentence Encoder

In [ ]:
import tensorflow as tf
# import tensorflow_hub as hub
import tensorflow.keras
from tensorflow.keras import backend as K

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import metrics as kmetrics
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

# Bert
import transformers
from transformers import *

os.environ["TF_KERAS"]='1'

In [ ]:
print(tf.__version__)
print(tensorflow.__version__)
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

In [ ]:
import tensorflow_hub as hub
import tensorflow_text

In [ ]:
os.environ["TFHUB_CACHE_DIR"] = "C:/Users/boumb/AppData/Local/Temp/tfhub_modules"


In [ ]:
embed = hub.KerasLayer(hub.load('https://tfhub.dev/google/universal-sentence-encoder/4'))
""" 
embed = hub.KerasLayer("D:\01-OpenClassRoom\PROJET\PROJET06\data\use")
URL = 'https://tfhub.dev/google/universal-sentence-encoder/4'
embed = hub.KerasLayer(URL, trainable=True)
embed = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')
"""

In [ ]:
def feature_USE_fct(sentences, b_size):
    batch_size = b_size
    time1 = time.time()

    for step in range(len(sentences)//batch_size) :
        idx = step*batch_size
        feat = embed(sentences[idx:idx+batch_size])

        if step ==0 :
            features = feat
        else :
            features = np.concatenate((features,feat))

    time2 = np.round(time.time() - time1,0)
    return features

In [ ]:
batch_size = 1050
sentences = corpus

In [ ]:
features_USE = feature_USE_fct(sentences, batch_size)

### Reduction des dimensions
#### ACP 

In [ ]:
# Create model alrdeady down pca = decomposition.PCA(n_components=2)
# Fit wor2sec
pca.fit(features_USE)
#print (pca.explained_variance_ratio_.cumsum()) # dans l autre sesn
X_trans_USE = pca.transform(features_USE)
print(pca.explained_variance_ratio_)

In [ ]:
df_X_trans_USE= pd.DataFrame(X_trans_USE)
df_X_trans_USE

In [ ]:
# visu
ax = plt.figure()
plt.title("PCA USE",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels=list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.scatter(df_X_trans_USE[0],df_X_trans_USE[1], c = df["cat_number"], cmap='Set1') 
plt.show()

#### T-SNE

In [ ]:
# test 1
#X_embedded_USE = TSNE(n_components=2, learning_rate='auto',init='pca', perplexity=100, random_state = 2023).fit_transform(features_USE)
#X_embedded_USE.shape
tsne_result_USE = tsne.fit_transform(features_USE)
tsne_result_USE.shape # (1050, 2)

In [ ]:
# Plot the result of our TSNE with the label color coded
tsne_result_USE_df = pd.DataFrame({'tsne_1': tsne_result_USE[:,0], 'tsne_2': tsne_result_USE[:,1], 'label': df["cat_number"]})
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_USE_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_USE.min()-5, tsne_result_USE.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.title("t-SNE USE")

### Clustering  kmeans, k=7

In [ ]:
# Initialize the clusterer with n_clusters value and a random generator :model1_kmn7 = KMeans(n_clusters= 7, n_init="auto", random_state=10)
# fit
y_labels_model1_kmn3_USE = model1_kmn7.fit_predict(tsne_result_USE_df)

In [ ]:
y_labels_model1_kmn3_USE

In [ ]:
# 3 cluster
y_USE_km7 = model1_kmn7.fit(tsne_result_USE)
y_USE_km7

In [ ]:
tsne_result_USE_df['Labels_KMN7'] = pd.DataFrame(y_USE_km7.labels_)
tsne_result_USE_df['Labels_KMN7'].unique()

In [ ]:
tsne_result_USE_df

In [ ]:
# Plot the result of TSNE with color as cluster results
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='Labels_KMN7', data=tsne_result_USE_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_USE.min()-5, tsne_result_USE.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
fig, ax = plt.subplots(1)
sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_USE_df, ax=ax,s=120,alpha=0.4)
lim = (tsne_result_USE.min()-5, tsne_result_USE.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
ARI_USE, X_tsne_USE, labels_USE = ARI_fct(features_USE) # devrait trouver entre 0.4 et 0.5

In [ ]:
TSNE_visu_fct(X_tsne_USE, df["cat_number"], labels_USE, ARI_USE,"USE") 

# IMAGE

In [ ]:
list_photos = list(df_text['image'])
len(list_photos)

In [ ]:
df_text.columns

In [ ]:
list_col_image = ['image','category']
df_image = df_text[list_col_image]
df_image

## Creation bag of feature

### Création des labels des photos à partir du nom des categories

In [ ]:
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
df_image["label"] = le.fit_transform(df_image["category"])
df_image

In [ ]:
set(df_image["label"])

In [ ]:
df_image.groupby("label").count() # 150 image par categorie 

### Affichage d'exemples d'images par label

In [ ]:
list_labels = list(set(df_image["category"]))
list_labels

In [ ]:
list_photos = list(df_image["image"])

In [ ]:
from matplotlib.image import imread
path = "data/Images/"
def list_fct(name) :
    list_image_name = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]==name]
    return list_image_name
#ou ala main
list_home = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Home Furnishing']
list_computers = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Computers']
list_baby = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Baby Care']
list_watches = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Watches']
list_kitchen = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Beauty and Personal Care']
list_decor = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Home Decor & Festive Needs']
list_kitchen = [list_photos[i] for i in range(len(df_image)) if df_image["category"][i]== 'Kitchen & Dining']

In [ ]:
list_fct("Computers")

In [ ]:
for name in list_labels :
    print(name)
    for i in range(3):
        plt.subplot(130 + 1 + i)
        filename = path + list_fct(name)[i+10]
        image = imread(filename)
        plt.imshow(image)
    plt.show()

## Détermination et affichage des descripteurs SIFT

* L'image contient  descripteurs
* Chaque descripteur est un vecteur de longueur 150

In [ ]:
import time, cv2
# exemple sur une image
image = cv2.imread(path+list_photos[1],0) # convert in gray
image = cv2.equalizeHist(image)   # equalize image histogram
converted_img = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
dst = cv2.fastNlMeansDenoisingColored(converted_img, None, 10, 10, 7, 15)


In [ ]:
plt.subplot(121), plt.imshow(image)
plt.subplot(122), plt.imshow(dst)

In [ ]:
sift = cv2.xfeatures2d.SIFT_create()
kp, des = sift.detectAndCompute(dst, None)
img=cv2.drawKeypoints(dst,kp,dst)
plt.imshow(img)
plt.show()
print("Descripteurs : ", des.shape)
print()
print(des)

## SIFT : Pré-traitement des images via SIFT

### Créations des descripteurs de chaque image
* Pour chaque image passage en gris et equalisation
* création d'une liste de descripteurs par image ("sift_keypoints_by_img") qui sera utilisée pour réaliser les histogrammes par image
* création d'une liste de descripteurs pour l'ensemble des images ("sift_keypoints_all") qui sera utilisé pour créer les clusters de descripteurs

In [ ]:
# identification of key points and associated descriptors
import time, cv2
sift_keypoints = []
temps1 = time.time()
sift = cv2.xfeatures2d.SIFT_create(500)
#sift = cv2.ORB_create(500)
 
for image_num in range(len(list_photos)) :
    if image_num%100 == 0 : print(image_num)
    image = cv2.imread(path+list_photos[image_num],0) # convert in gray
    #image = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    res = cv2.equalizeHist(image)   # equalize image histogram
    kp, des = sift.detectAndCompute(res, None)
    sift_keypoints.append(des)

#sift_keypoints_by_img = np.asarray(sift_keypoints)
sift_keypoints_all = np.concatenate(sift_keypoints, axis=0)

print()
print("Nombre de descripteurs : ", sift_keypoints_all.shape)

duration1=time.time()-temps1
print("temps de traitement SIFT descriptor : ", "%15.2f" % duration1, "secondes")

### Création des features des images
* Pour chaque image : 
   - prédiction des numéros de cluster de chaque descripteur
   - création d'un histogramme = comptage pour chaque numéro de cluster du nombre de descripteurs de l'image

In [ ]:
from sklearn import cluster, metrics

# Determination number of clusters

k = int(round(np.sqrt(len(sift_keypoints_all)),0))
print("Nombre de clusters estimés : ", k)
print("Création de",k, "clusters de descripteurs ...")

# Clustering
kmeans = cluster.MiniBatchKMeans(n_clusters=k, init_size=3*k, random_state=0)
kmeans.fit(sift_keypoints_all)

duration1=time.time()-temps1
print("temps de traitement kmeans : ", "%15.2f" % duration1, "secondes")

In [ ]:
# Creation of histograms (features)

def build_histogram(kmeans, des, image_num):
    res = kmeans.predict(des)
    hist = np.zeros(len(kmeans.cluster_centers_))
    nb_des=len(des)
    if nb_des==0 : print("problème histogramme image  : ", image_num)
    for i in res:
        hist[i] += 1.0/nb_des
    return hist


# Creation of a matrix of histograms
hist_vectors=[]

for i, image_desc in enumerate(sift_keypoints) :
    if i%100 == 0 : print(i)  
    hist = build_histogram(kmeans, image_desc, i) #calculates the histogram
    hist_vectors.append(hist) #histogram is the feature vector

im_features = np.asarray(hist_vectors)

duration1=time.time()-temps1
print("temps de création histogrammes : ", "%15.2f" % duration1, "secondes")


### Réductions de dimension

#### PCA
* La réduction PCA permet de créer des features décorrélées entre elles, et de diminuer leur dimension, tout en gardant un niveau de variance expliquée élevé (99%)
* L'impact est une meilleure séparation des données via le T-SNE et une réduction du temps de traitement du T-SNE

In [ ]:
from sklearn import manifold, decomposition

print("Dimensions dataset avant réduction PCA : ", im_features.shape)
pca2 = decomposition.PCA(n_components=2)
feat_pca2= pca2.fit_transform(im_features)
print("Dimensions dataset après réduction PCA : ", feat_pca2.shape)

In [ ]:
# visu
ax = plt.figure()
plt.title("PCA SIFT",fontweight="bold",fontsize=20)
ax.legend(handles=scatter.legend_elements()[0], labels=list_category,bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.scatter(feat_pca2[:,0],feat_pca2[:,1], c = df["cat_number"] , cmap='Set1') 
plt.xlabel ("PCA1 ("+str(np.round (pca.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca.explained_variance_ratio_[1]*100,2))+"%)")
plt.xlim(-0.02,0.1)
plt.ylim(-0.15,0.12)
plt.show()

#### T-SNE
* Réduction de dimension en 2 composantes T-SNE pour affichage en 2D des images

In [ ]:
from sklearn import manifold, decomposition
tsne2 = manifold.TSNE(n_components=2, perplexity=30, 
                     n_iter=2000, init='random', random_state=6)
X_tsne_image = tsne2.fit_transform(im_features)

df_tsne_image = pd.DataFrame(X_tsne_image[:,0:2], columns=['tsne1', 'tsne2'])
# df_tsne_result_image = pd.DataFrame({'tsne_1': tsne_result_USE[:,0], 'tsne_2': tsne_result_USE[:,1], 'label': df["cat_number"]})
df_tsne_image["label"] =  df["cat_number"]
print(df_tsne_image.shape)


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    x="tsne1", y="tsne2", hue="label", data=df_tsne_image, legend="brief",
    palette=sns.color_palette('tab10', n_colors=4), s=20, alpha=0.6)
plt.title('TSNE selon les vraies classes', fontsize = 30, pad = 35, fontweight = 'bold')
plt.xlabel('tsne1', fontsize = 26, fontweight = 'bold')
plt.ylabel('tsne2', fontsize = 26, fontweight = 'bold')
plt.legend(prop={'size': 14}) 

plt.show()


###  Création de clusters à partir du T-SNE

In [ ]:
from sklearn import cluster, metrics

cls = cluster.KMeans(n_clusters=7, random_state=6)
cls.fit(X_tsne_image)

df_tsne_image["cluster"] = cls.labels_
print(df_tsne_image.shape)


In [ ]:

plt.figure(figsize=(10,6))
sns.scatterplot(
    x="tsne1", y="tsne2",
    hue="cluster",
    palette=sns.color_palette('tab10', n_colors=7), s=50, alpha=0.6,
    data=df_tsne_image,
    legend="brief")

plt.title('TSNE selon les clusters', fontsize = 30, pad = 35, fontweight = 'bold')
plt.xlabel('tsne1', fontsize = 26, fontweight = 'bold')
plt.ylabel('tsne2', fontsize = 26, fontweight = 'bold')
plt.legend(prop={'size': 14}) 

plt.show()

labels_vrai = df_image["label"]
print("ARI : ", metrics.adjusted_rand_score(labels_vrai, df_tsne_image["cluster"]))

In [ ]:
ARI_SIFT, X_tsne_SIFT, labels_SIFT = ARI_fct(im_features) # doit trouver entre 0.4 et 0.5 a revoir


In [ ]:
TSNE_visu_fct(X_tsne_SIFT, labels_vrai, labels_SIFT, ARI_SIFT," SIFT") 

### Analyse par classes
* La matrice de confusion doit être transformée pour mettre avoir en colonne le même ordre des catégories supposées qu'en ligne
* Cette transformation atteint ses limites pour les 2 catégories "shine" et "sunshine" qui sont peu séparées (donc elle a été réalisée manuellement au lieu d'utiliser la fonction "argmax")

In [ ]:
df_tsne_image.groupby("cluster").count()["label"]

In [ ]:
conf_mat = metrics.confusion_matrix(list(df_image["label"]), list(labels_SIFT))
print(conf_mat)

In [ ]:
set(df_image["label"]),set(labels_bert) #ok

In [ ]:
set(np.argmax(conf_mat, axis=0)) # should be 0 to 6

In [ ]:
def conf_mat_transform(y_true,y_pred) :
    conf_mat = metrics.confusion_matrix(y_true,y_pred)
    #corresp = np.argmax(conf_mat, axis=0) pb avec la fonction argmax
    corresp = [0,1,2,3, 4, 5,6] 
    print ("Correspondance des clusters : ", corresp)
    
    labels = pd.Series(y_true, name="y_true").to_frame()
    labels['y_pred'] = y_pred
    labels['y_pred_transform'] = labels['y_pred'].apply(lambda x : corresp[x]) 
    
    return labels['y_pred_transform']

cls_labels_transform = conf_mat_transform(labels_vrai,labels_SIFT )
conf_mat2 = metrics.confusion_matrix(labels_vrai, cls_labels_transform)
print(conf_mat2)
print()
print(metrics.classification_report(labels_vrai, cls_labels_transform))

In [ ]:
df_cm = pd.DataFrame(conf_mat2, index = [label for label in list_labels],
                  columns = [i for i in "0123456"])
plt.figure(figsize = (6,4))
sns.heatmap(df_cm, annot=True, cmap="Blues")

## CNN : Transfer Learning in Keras VG16 preentrainé

### Chargement du modele avec la classe VG16, Pre-Trained Model as Classifier
Next, the model can be loaded and a prediction made.
When loading a given model, the “include_top” argument can be set to False, in which case the fully-connected output layers of the model used to make predictions is not loaded, allowing a new output layer to be added and trained.
model = VGG16(include_top=False)

In [ ]:
from keras.applications.vgg16 import VGG16
from keras.layers import Dense
from keras import Model
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input
from keras.applications.vgg16 import decode_predictions
from keras.applications.vgg16 import decode_predictions
# utilisation du model VG16
model = VGG16()
from pickle import dump

In [ ]:
# summarize the model
model.summary()

In [ ]:
# prepare an image
# load an image from file
image = load_img(path+list_photos[1], target_size=(224, 224))
# convert the image pixels to a numpy array
image = img_to_array(image)
# reshape data for the model
image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
# prepare the image for the VGG model
image = preprocess_input(image)

In [ ]:

# predict the probability across all output classes
y = model.predict(image)  # Prédir la classe de l'image (parmi les 1000 classes d'ImageNet)
len(y)

In [ ]:
# convert the probabilities to class labels
label = decode_predictions(y)
# retrieve the most likely result, e.g. highest probability
label = label[0][0]
# print the classification
print('%s (%.2f%%)' % (label[1], label[2]*100))

In [ ]:
list_photos[0:10]

In [ ]:
# Pour toutes les image de la liste
list_prediction = []
for i in range(len(list_photos[0:10])):

    # load an image from file
    image = load_img(path+list_photos[i], target_size=(224, 224))
    # convert the image pixels to a numpy array
    image = img_to_array(image)
    # reshape data for the model
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    # prepare the image for the VGG model
    image = preprocess_input(image)

    y = model.predict(image)
    # convert the probabilities to class labels
    label = decode_predictions(y)
    # retrieve the most likely result, e.g. highest probability
    label = label[0][0]
    label2 = label[1]
    list_prediction.append(label2)
    # print the classification
    #print(list_prediction)
    print('%s (%.2f%%)' % (label[1], label[2]*100))

In [ ]:
list_prediction

### Extraction des features

In [ ]:
model = VGG16()
# remove the output layer
model = Model(inputs=model.inputs, outputs=model.layers[-2].output)

In [ ]:
list_photos[1]

In [ ]:
# prepare an image
# load an image from file
image = load_img(path+list_photos[1], target_size=(224, 224))
# convert the image pixels to a numpy array
image = img_to_array(image)
# reshape data for the model
image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
# prepare the image for the VGG model
image = preprocess_input(image)

In [ ]:
# get extracted features
features = model.predict(image)
print(features.shape)


In [ ]:
# Pour toutes les image de la liste
list_feature = []
for i in range(len(list_photos)):

    # load an image from file
    image = load_img(path+list_photos[i], target_size=(224, 224))
    # convert the image pixels to a numpy array
    image = img_to_array(image)
    # reshape data for the model
    image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
    # prepare the image for the VGG model
    image = preprocess_input(image)
    features = model.predict(image)
    list_feature.extend(features.tolist()) 


In [ ]:
array_feature = np.array(list_feature)
array_feature.shape

### PCA

In [ ]:
from sklearn import manifold, decomposition

print("Dimensions dataset avant réduction PCA : ",array_feature.shape)
feat_pca2_cnn = pca2.fit_transform(list_feature)
print("Dimensions dataset après réduction PCA : ", feat_pca2.shape)

In [ ]:
feat_pca2_cnn[:,0]

In [ ]:
np.unique(labels_vrai)

In [ ]:
np.array(labels_vrai)

In [ ]:
df_X_trans_cnn = pd.DataFrame(feat_pca2_cnn)
df_X_trans_cnn["label"] = labels_vrai
# visu
ax = plt.figure()
# plt.scatter(df_X_trans_cnn[0],df_X_trans_cnn[1], c = labels_vrai, label = df_X_trans_cnn["label"],cmap='Set1' )
scatter = plt.scatter(feat_pca2_cnn[:,0],feat_pca2_cnn[:,1],c = np.array(labels_vrai),label=np.array(df_text['category']) ,cmap='Set1' )
plt.title("PCA CNN VGG16")
plt.xlabel ("PCA1 ("+str(np.round (pca2.explained_variance_ratio_[0]*100,2))+"%)")
plt.ylabel ("PCA2 ("+str(np.round (pca2.explained_variance_ratio_[1]*100,2))+"%)")
ax.legend( *scatter.legend_elements(),bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.show()

### t-SNE sur feature extraite avec cnn preentrainé

In [ ]:
X_tsne_image_VV16 = tsne2.fit_transform(np.array(list_feature))
df_tsne_image_VV16 = pd.DataFrame(X_tsne_image_VV16[:,0:2], columns=['tsne1', 'tsne2'])
print(df_tsne_image_VV16.shape)


###  Création de clusters à partir du T-SNE

In [ ]:
from sklearn import cluster, metrics

cls = cluster.KMeans(n_clusters=7, random_state=6)
cls.fit(X_tsne_image)

df_tsne_image["cluster"] = cls.labels_
print(df_tsne_image.shape)


###  Affichage des images selon clusters et calcul ARI de similarité catégories images / clusters
* Le score ARI de 0.15 reste faible

In [ ]:

plt.figure(figsize=(10,6))
sns.scatterplot(
    x="tsne1", y="tsne2",
    hue="cluster",
    palette=sns.color_palette('tab10', n_colors=7), s=50, alpha=0.6,
    data=df_tsne_image,
    legend="brief")
plt.title('TSNE selon les clusters', fontsize = 30, pad = 35, fontweight = 'bold')
plt.xlabel('tsne1', fontsize = 26, fontweight = 'bold')
plt.ylabel('tsne2', fontsize = 26, fontweight = 'bold')
plt.legend(prop={'size': 14},bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0) 
plt.show()

labels = df_image["label"]
print("ARI : ", metrics.adjusted_rand_score(labels, cls.labels_))

In [ ]:
ARI_CNN, X_tsne_CNN, labels_CNN = ARI_fct(np.array(list_feature)) # devrait trouver entre 0.4 et 0.5


In [ ]:
TSNE_visu_fct(X_tsne_CNN, labels_vrai, labels_CNN, ARI_CNN,"IMAGE CNN") 

In [ ]:
cls_labels_transform = conf_mat_transform(labels_vrai,labels_CNN )
conf_mat3 = metrics.confusion_matrix(labels_vrai, cls_labels_transform)
print(conf_mat3)
print(metrics.classification_report(labels_vrai, cls_labels_transform))


In [ ]:
df_cm = pd.DataFrame(conf_mat3, index = [label for label in list_labels],
                  columns = [i for i in "0123456"])
plt.figure(figsize = (6,4))
sns.heatmap(df_cm, annot=True, cmap="Blues")